# Utilities, Security and Miscellaneous Functions

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_customers    = spark.table("samples.bakehouse.sales_customers")

## Problem 1 - String Utilities: concat_ws, ltrim, rtrim, levenshtein

Use df_customers to build a full name from first and last name columns using `F.concat_ws`.
Apply `F.ltrim` and `F.rtrim` on a test string with whitespace padding.
Use `F.levenshtein` to compute edit distance between two string columns.

**Expected output columns:** `customerID`, `full_name`, `edit_distance`

In [ ]:
# Problem 1 - write your solution here
# result_1 = None  # replace this

In [ ]:
assert result_1 is not None
assert hasattr(result_1, "columns")
cols = [c.lower() for c in result_1.columns]
assert "full_name" in cols
assert "edit_distance" in cols
cnt = result_1.count()
assert cnt > 0
print(f"Problem 1 passed ({cnt} rows)")

## Problem 2 - Hashing for Data Masking: md5, sha2, hash, crc32

Use `F.md5`, `F.sha2(col, 256)`, `F.hash`, and `F.crc32` on a sensitive column in df_customers.
- `F.md5(col)` returns a 32-char hex string
- `F.sha2(col, 256)` returns a 64-char hex string
- `F.hash(col)` returns a numeric hash (int)
- `F.crc32(col)` returns a CRC32 checksum as a long integer

**Expected output columns:** `customerID`, `email_md5`, `email_sha256`, `email_hash`, `email_crc32`

In [ ]:
# Problem 2 - write your solution here
# result_2 = None  # replace this

In [ ]:
assert result_2 is not None
cols = [c.lower() for c in result_2.columns]
assert 'email_md5' in cols
assert 'email_sha256' in cols
assert 'email_crc32' in cols
row = result_2.first()
assert len(row['email_md5']) == 32
assert len(row['email_sha256']) == 64
print('Problem 2 passed')

## Problem 3 - Encoding: base64 and unbase64

Encode a string column to base64 and decode it back to verify the round-trip.
Create a small test DataFrame, apply `F.encode` to get binary, then `F.base64` to encode, then `F.unbase64` + `F.decode` to get back the original string.

**Expected output columns:** `text`, `b64`, `decoded`

In [ ]:
# Problem 3 - write your solution here
# result_3 = None  # replace this

In [ ]:
assert result_3 is not None
cols = [c.lower() for c in result_3.columns]
assert "text" in cols
assert "b64" in cols
assert "decoded" in cols
print("Problem 3 passed")

## Problem 4 - Row Metadata: monotonically_increasing_id and spark_partition_id

Use `F.monotonically_increasing_id()` to assign a unique row ID and `F.spark_partition_id()` to show which partition each row belongs to.
Repartition df_transactions to 4 partitions first.

**Expected output columns:** `transactionID`, `row_id`, `partition_id`

In [ ]:
# Problem 4 - write your solution here
# result_4 = None  # replace this

In [ ]:
assert result_4 is not None
cols = [c.lower() for c in result_4.columns]
assert "row_id" in cols
assert "partition_id" in cols
partitions = {r["partition_id"] for r in result_4.select("partition_id").distinct().collect()}
assert len(partitions) <= 4
print(f"Problem 4 passed ({len(partitions)} partitions)")

## Problem 5 - Global Temp Views: createGlobalTempView

Register df_transactions as a global temporary view using `createGlobalTempView`.
Global temp views are accessible across sessions under the `global_temp` database.
Query it using `spark.sql`.

**Expected output columns:** `total_rows` (from a COUNT query)

In [ ]:
# Problem 5 - write your solution here
# result_5 = None  # replace this

In [ ]:
assert result_5 is not None
cnt = result_5.collect()[0][0]
assert cnt > 0
print(f"Problem 5 passed ({cnt} rows in global temp view)")

## Problem 6 - crossJoin, exceptAll, intersectAll

Demonstrate three operations:
1. `crossJoin` - explicit method for cross product
2. `exceptAll` - like subtract but preserves duplicates
3. `intersectAll` - like intersect but preserves duplicates

Create small test DataFrames to show the difference.

In [ ]:
# Problem 6 - write your solution here
# result_6 = None  # replace this (the crossJoin result)

In [ ]:
assert result_6 is not None
cnt = result_6.count()
assert cnt == 9  # 3 x 3 cross join
print(f"Problem 6 passed (crossJoin: {cnt} rows)")

## Problem 7 - mapInPandas

Use `mapInPandas` to process each partition as a Pandas DataFrame.
This is useful for batch inference or complex per-partition transformations.
Normalize `totalPrice` within each partition.

**Expected output columns:** `transactionID`, `franchiseID`, `totalPrice`, `normalized_price`

In [ ]:
# Problem 7 - write your solution here
# result_7 = None  # replace this

In [ ]:
assert result_7 is not None
cols = [c.lower() for c in result_7.columns]
assert "normalized_price" in cols
from pyspark.sql import functions as F
max_np = result_7.agg(F.max("normalized_price")).collect()[0][0]
min_np = result_7.agg(F.min("normalized_price")).collect()[0][0]
assert max_np <= 1.0
assert min_np >= 0.0
print("Problem 7 passed")

## Problem 8 - Math Utilities and repartitionByRange

Apply math functions to `totalPrice` in df_transactions:
- `F.sqrt(col)` - square root
- `F.abs(col)` - absolute value
- `F.signum(col)` - sign (-1, 0, or 1)
- `F.nullif(col1, col2)` - returns null if col1 == col2, otherwise col1
- `F.nanvl(col1, col2)` - returns col1 if not NaN, else col2 (for float/double columns)

Also use `repartitionByRange(4, col)` which partitions data so each partition covers a price range.

**Expected output columns:** `transactionID`, `totalPrice`, `sqrt_price`, `sign_price`, `nanvl_example`

In [ ]:
# Problem 8 - write your solution here
# result_8 = None  # replace this

In [ ]:
assert result_8 is not None
cols = [c.lower() for c in result_8.columns]
assert 'sqrt_price' in cols
assert 'sign_price' in cols
assert 'nanvl_example' in cols
from pyspark.sql import functions as F
min_sqrt = result_8.agg(F.min('sqrt_price')).collect()[0][0]
assert min_sqrt >= 0
print('Problem 8 passed')